In [1]:
import time

In [2]:
start_notebook = time.time()

In [3]:
import warnings
warnings.filterwarnings("ignore")

# 1. Parameters

In [4]:
name_dataset = 'AG_News'
name_model = 'gpt-4o'
mode = 'few'
seed = 1
part = 4

In [5]:
path_open = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post12/df_test_{part}.csv'

In [6]:
path_save = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/03.Inference/{name_dataset}/post13/df_test_{part}.csv'

In [7]:
path_credentials = f'/content/drive/MyDrive/Finetuned-encoders-LLM-Prompting/credentials/OPENAI_API_KEY.json'

# 2. Load Environment

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import json
import requests
import pandas as pd
from openai import OpenAI

In [10]:
with open(path_credentials, "r") as f:
    credentials = json.load(f)

In [11]:
OPENAI_API_KEY = credentials["OPENAI_API_KEY"]

In [12]:
client = OpenAI(api_key=OPENAI_API_KEY)

# 3. Functions

In [13]:
def few_shot_prompt(text):

    intro = (
        "Classify the topic of the following news article from the AG News dataset.\n"
        "Respond only with a single digit according to the category:\n"
        "0 = World\n"
        "1 = Sports\n"
        "2 = Business\n"
        "3 = Sci/Tech\n"
        "Return only the digit (no words, no punctuation).\n"
    )

    few_shots = (
        "\nHere are some examples:\n\n"
        "Example 1:\n"
        "Article: \"Kerry leading Bush in key swing states (AFP) AFP - Although polls show the US presidential race a virtual dead heat, Democrat John Kerry appears to be gaining an edge over George W. Bush among the key states that could decide the outcome.\"\n"
        "Label: 0\n\n"

        "Example 2:\n"
        "Article: \"Colander Misses Chance to Emulate Jones  ATHENS (Reuters) - But for a decision that enraged her coach, LaTasha Colander might have been the Marion Jones of the Athens Olympics.\"\n"
        "Label: 1\n\n"

        "Example 3:\n"
        "Article: \"Oil and Economy Cloud Stocks' Outlook  NEW YORK (Reuters) - Soaring crude prices plus worries about the economy and the outlook for earnings are expected to hang over the stock market next week during the depth of the summer doldrums.\"\n"
        "Label: 2\n\n"

        "Example 4:\n"
        "Article: \"Apple to open second Japanese retail store this month (MacCentral) MacCentral - Apple Computer Inc. will open its second Japanese retail store later this month in the western Japanese city of Osaka, it said Thursday.\"\n"
        "Label: 3\n\n"
    )

    target = (
        "Now classify the following article:\n"
        f"Article: \"{text}\"\n"
        "Label:"
    )

    return intro + few_shots + target

In [ ]:
def predict_label(text, prompt):

    try:

        start_time = time.perf_counter()
        first_token_time = None
        output_text = ""

        stream = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            top_p=1.0,
            seed=seed,
            stream=True,
            stream_options={"include_usage": True}
        )

        for chunk in stream:
            if first_token_time is None and len(chunk.choices) > 0 and chunk.choices[0].delta.content:
                first_token_time = time.perf_counter()

            if len(chunk.choices) > 0 and chunk.choices[0].delta.content:
                output_text += chunk.choices[0].delta.content

            if chunk.usage is not None:
                usage = chunk.usage

        end_time = time.perf_counter()

        return {
            "prediction": output_text.strip(),
            "latency_ms": (end_time - start_time) * 1000,
            "ttft_ms": (
                (first_token_time - start_time) * 1000
                if first_token_time else None
            ),
            "input_tokens": usage.prompt_tokens,
            "output_tokens": usage.completion_tokens
        }
    
    except:
        
        return {
            "prediction": '-1',
            "latency_ms": '-',
            "ttft_ms": '-',
            "input_tokens": '-',
            "output_tokens": '-'
        }

# 4. Load Dataset

In [15]:
df = pd.read_csv(path_open)

In [16]:
df.shape

(760, 36)

In [17]:
pred_label = []
pred_latency = []
pred_ttft = []
pred_input_tokens = []
pred_output_tokens = []

In [18]:
for i in range(len(df)):

  text = df['text'].iloc[i]
  prompt = few_shot_prompt(text)
  output = predict_label(text, prompt)

  pred_label.append(int(output['prediction'][0]))
  pred_latency.append(output['latency_ms'])
  pred_ttft.append(output['ttft_ms'])
  pred_input_tokens.append(output['input_tokens'])
  pred_output_tokens.append(output['output_tokens'])

  if (i % 10) == 0:
    print(i)

0
10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
230
240
250
260
270
280
290
300
310
320
330
340
350
360
370
380
390
400
410
420
430
440
450
460
470
480
490
500
510
520
530
540
550
560
570
580
590
600
610
620
630
640
650
660
670
680
690
700
710
720
730
740
750


In [19]:
df[f'{name_model}-{mode}-seed-{seed}-label'] = pred_label
df[f'{name_model}-{mode}-seed-{seed}-latency'] = pred_latency
df[f'{name_model}-{mode}-seed-{seed}-ttft'] = pred_ttft
df[f'{name_model}-{mode}-seed-{seed}-input-tokens'] = pred_input_tokens
df[f'{name_model}-{mode}-seed-{seed}-output-tokens'] = pred_output_tokens

In [20]:
df[f'{name_model}-{mode}-seed-{seed}-label'].value_counts()

,count
gpt-4o-few-seed-1-label,
2,219
3,196
1,189
0,156


In [21]:
df[f'{name_model}-{mode}-seed-{seed}-latency'].describe()

,gpt-4o-few-seed-1-latency
count,760.000000
mean,395.329206
std,274.147315
min,246.357556
25%,306.010962
50%,343.960647
75%,400.425649
max,5313.972168


In [22]:
df[f'{name_model}-{mode}-seed-{seed}-ttft'].describe()

,gpt-4o-few-seed-1-ttft
count,760.000000
mean,391.612884
std,274.180978
min,245.714152
25%,302.973788
50%,340.536064
75%,394.609237
max,5313.135507


In [23]:
df[f'{name_model}-{mode}-seed-{seed}-input-tokens'].describe()

,gpt-4o-few-seed-1-input-tokens
count,760.000000
mean,358.118421
std,17.602002
min,326.000000
25%,348.000000
50%,356.000000
75%,365.000000
max,513.000000


In [24]:
df[f'{name_model}-{mode}-seed-{seed}-output-tokens'].describe()

,gpt-4o-few-seed-1-output-tokens
count,760.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


# 5. Save Dataset

In [25]:
df.to_csv(path_save, index = False)

# 6. Execution Time

In [26]:
end_notebook = time.time()

In [27]:
delta_notebook = end_notebook - start_notebook
hours_notebook, rem_notebook = divmod(delta_notebook, 3600)
minutes_notebook, seconds_notebook = divmod(rem_notebook, 60)

print(f"Execution Notebook: {int(hours_notebook)}h {int(minutes_notebook)}m {seconds_notebook:.2f}s")

Execution Notebook: 0h 5m 6.02s
